In [1]:
from importlib import reload

import perceptual as pt # torch manipulations
import oklab as pn # numpy manipulations



import numpy as np
import torch



def tensorify(nparr):
    return torch.tensor(nparr).permute([2,0,1]).unsqueeze(0)
def untensorify(tensor):
    return tensor.squeeze().permute([1,2,0]).detach().cpu().numpy()

def arr_diff(nparr, tensor):
    """returns the l2 difference between a HWC numpy array
    and a BCHW torch tensor"""
    # tensor_reshaped = tensor.squeeze().permute([1,2,0])
    # tensor_np = tensor_reshaped.detach().cpu().numpy()
    tensor_np = untensorify(tensor)

    diff = tensor_np - nparr
    return ((diff**2)**0.5).mean()


# numpy example array: (HWC)
rgbn = np.random.uniform(0, 1, ((5,5,3)))

# torch example tensor (BCHW)
# rgbt = torch.tensor(rgbn).permute([2,0,1]).unsqueeze(0)
rgbt = tensorify(rgbn)

print(f'{rgbn.shape=}')
print(f'{rgbt.shape=}')

print(f'{arr_diff(rgbn, rgbt):.8f}')

rgbn.shape=(5, 5, 3)
rgbt.shape=torch.Size([1, 3, 5, 5])
0.00000000


In [2]:
### convert both to oklab:

labn = pn.srgb_to_oklab(rgbn)
labt = pt.srgb_to_oklab(rgbt)

print(f'{labn.shape=}')
print(f'{labt.shape=}')
print(f'{arr_diff(labn, labt):.8f}')

labn.shape=(5, 5, 3)
labt.shape=torch.Size([3, 5, 3])


ValueError: operands could not be broadcast together with shapes (5,3,3) (5,5,3) 

In [ ]:
### apply a random perturbation in lab space:

lab_delta = np.random.normal(0, 0.02, ((5,5,3)))

labn2 = labn + lab_delta
labt2 = labt + tensorify(lab_delta)

print(f'{labn2.shape=}')
print(f'{labt2.shape=}')
print(f'{arr_diff(labn2, labt2):.8f}')

In [ ]:
### convert perturbed arrays back to rgb space

rgbn2 = pn.oklab_to_srgb(labn2, clip=False)
rgbt2 = pt.oklab_to_srgb(labt2, clip=False)

print(f'{rgbn2.shape=}')
print(f'{rgbt2.shape=}')
print(f'{arr_diff(rgbn2, rgbt2):.8f}')

assert arr_diff(rgbn2, rgbt2) <= 1e-12 # error is not exactly 0 but close enough